*Migrated to NotebookSession API in Phase 4b — see notebooks/UTIL_README.md*

# ADP1 BERDL Fold Change Flux Analysis

## Objectives

This notebook performs flux analysis based on proteomics fold changes between strains,
using the **BERDL fitness-constrained flux** as the reference distribution (from ADP1BERDLFitnessFluxFitting).

### Analysis Goals:
1. Load proteomics data (log2 format) and average replicates
2. Load reference flux distribution from BERDL fitness-constrained pyruvate solution
3. For each strain (target condition), compute fold changes relative to ADP1 (reference)
4. Use MSExpression.fit_flux_to_proteomics_fold_change_data to fit fluxes
5. Analyze which protein fold changes could/could not be implemented
6. Compare flux changes across strains

### Strains (Target Conditions):
- **ACN2586**: Initial construct
- **ACN2821**: Evolved strain
- **ACN3425, ACN3427, ACN3429, ACN3430**: Additional strain variants

### Reference Condition:
- **ADP1**: Wild-type Acinetobacter baylyi ADP1

### Reference Flux:
- **BERDL fitness-constrained flux** on pyruvate media (from ADP1BERDLFitnessFluxFitting)
- This differs from ADP1FoldChangeAnalysis which uses the lactate condition from ADP1MutantPhenotypeAnalysis

## Load Proteomics Data and Average Replicates

This step:
1. Loads proteomics data from Excel spreadsheet (log2 format)
2. Identifies all strain conditions including ADP1 reference
3. Averages replicates while keeping data in log2 format
4. Saves averaged expression data for subsequent analysis

In [1]:
%run util.py


# Define all strain conditions (including ADP1 reference)
all_strains = [
    "Pyruvate_ACN2586_DgoA_2025",
    "Pyruvate_ACN2821_DgoA_2025",
    "Pyruvate_ACN3425_DgoA_2025",
    "Pyruvate_ACN3427_DgoA_2025",
    "Pyruvate_ACN3429_DgoA_2025",
    "Pyruvate_ACN3430_DgoA_2025",
    "Pyruvate_ADP1_DgoA_2025",
]

# Reference condition (ADP1 wild-type)
reference_condition = "Pyruvate_ADP1_DgoA_2025"
target_conditions = [s for s in all_strains if s != reference_condition]

print(f"Reference condition: {reference_condition}")
print(f"Target conditions: {target_conditions}")

# Ingest raw proteomics from Excel via session.vectors

# Register the external proteomics dataset
ext = ExternalDataset(
    id="berdl_proteomics_2025",
    source="collaborator",
    organism="Acinetobacter baylyi ADP1",
    description="UGA Proteomics DgoA strains May 2025 — pyruvate media, log2 format",
)
session.experiments.register_external(ext)

# Read raw Excel as DataFrame
raw_df = pd.read_excel(
    "data/ASCR_UGA_Proteomics_DgoA_add_strains_2025_pyruvate.xlsx",
    sheet_name="Imputed",
    index_col="ACIAD",
)

# Translate gene IDs (RefSeq -> old format) if mapping file exists
gene_mapping_file = "data/ADP1Genes.csv"
if os.path.exists(gene_mapping_file):
    raw_df = translate_expression_gene_ids(raw_df, gene_mapping_file)
    print(f"Translated gene IDs using {gene_mapping_file}")

print(f"Total features (genes): {len(raw_df)}")

# Average replicates for each strain (columns matching strain prefix)
averaged_data = {}
for strain in all_strains:
    strain_cols = [c for c in raw_df.columns if c.startswith(strain)]
    if strain_cols:
        averaged_data[strain] = raw_df[strain_cols].mean(axis=1)
    else:
        # Try exact match
        if strain in raw_df.columns:
            averaged_data[strain] = raw_df[strain]

averaged_df = pd.DataFrame(averaged_data)
print(f"\nAfter averaging replicates:")
print(f"  Conditions: {list(averaged_df.columns)}")
print(f"  Shape: {averaged_df.shape}")

# Save strain info and averaged data via session.cache
session.cache.save("berdl_fc_strains", {
    "all_strains": all_strains,
    "reference_condition": reference_condition,
    "target_conditions": target_conditions,
})
session.cache.save("berdl_fc_averaged_expression", averaged_df.to_dict())

print("\nSaved strain info and averaged expression data")

[KBUtilLib] Failed to import rcsb_pdb_utils: ModuleNotFoundError: No module named 'aiohttp'


Reference condition: Pyruvate_ADP1_DgoA_2025
Target conditions: ['Pyruvate_ACN2586_DgoA_2025', 'Pyruvate_ACN2821_DgoA_2025', 'Pyruvate_ACN3425_DgoA_2025', 'Pyruvate_ACN3427_DgoA_2025', 'Pyruvate_ACN3429_DgoA_2025', 'Pyruvate_ACN3430_DgoA_2025']
Translated gene IDs using data/ADP1Genes.csv
Total features (genes): 2383

After averaging replicates:
  Conditions: ['Pyruvate_ACN2586_DgoA_2025', 'Pyruvate_ACN2821_DgoA_2025', 'Pyruvate_ACN3425_DgoA_2025', 'Pyruvate_ACN3427_DgoA_2025', 'Pyruvate_ACN3429_DgoA_2025', 'Pyruvate_ACN3430_DgoA_2025', 'Pyruvate_ADP1_DgoA_2025']
  Shape: (2383, 7)

Saved strain info and averaged expression data


/home/chenry/VirtualEnvironments/kbu.nb-adp1notebooks-py3.10/lib/python3.10/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)


## Gene-Level Proteomics Fold Changes

This step:
1. Computes fold changes for each gene relative to ADP1 wild-type reference
2. Fold change = 2^(target_log2 - reference_log2), log2 FC = target_log2 - reference_log2
3. Saves six separate datacache files, one per strain: `berdl_fc_<strain>_vs_ADP1`
4. Each file is a dictionary: {gene_id: {"log2_fc": ..., "fold_change": ...}}

In [2]:
%run util.py


# Load averaged expression data
averaged_expression_data = session.cache.load("berdl_fc_averaged_expression")
strain_info = session.cache.load("berdl_fc_strains")

# Build DataFrame from saved data
expr_df = pd.DataFrame.from_dict(averaged_expression_data)

reference_condition = strain_info["reference_condition"]
target_conditions = strain_info["target_conditions"]

# Compute and save gene-level fold changes for each strain vs ADP1
for target in target_conditions:
    short_name = target.replace("Pyruvate_", "").replace("_DgoA_2025", "")

    strain_fc = {}
    for gene_id in expr_df.index:
        ref_val = expr_df.loc[gene_id, reference_condition]
        target_val = expr_df.loc[gene_id, target]

        # Both values are log2; fold change = 2^(target - reference)
        if pd.notna(ref_val) and pd.notna(target_val):
            log2_fc = target_val - ref_val
            linear_fc = 2 ** log2_fc
            strain_fc[gene_id] = round(float(linear_fc), 4)

    # Save individual file per strain
    cache_key = f"berdl_fc_{short_name}_vs_ADP1"
    session.cache.save(cache_key, strain_fc)

    # Summary stats
    values = list(strain_fc.values())
    up_2x = sum(1 for v in values if v >= 2.0)
    down_2x = sum(1 for v in values if v <= 0.5)

    print(f"{short_name}: {len(strain_fc)} genes -> saved as '{cache_key}'")
    print(f"  Mean FC: {np.mean(values):.3f}, Median: {np.median(values):.3f}")
    print(f"  Up >=2x: {up_2x}, Down <=0.5x: {down_2x}")

print(f"\nSaved 6 separate fold change files")

ACN2586: 2383 genes -> saved as 'berdl_fc_ACN2586_vs_ADP1'
  Mean FC: 1.491, Median: 1.006
  Up >=2x: 178, Down <=0.5x: 282
ACN2821: 2383 genes -> saved as 'berdl_fc_ACN2821_vs_ADP1'
  Mean FC: 1.291, Median: 0.935
  Up >=2x: 71, Down <=0.5x: 239
ACN3425: 2383 genes -> saved as 'berdl_fc_ACN3425_vs_ADP1'
  Mean FC: 1.338, Median: 1.000
  Up >=2x: 75, Down <=0.5x: 229
ACN3427: 2383 genes -> saved as 'berdl_fc_ACN3427_vs_ADP1'
  Mean FC: 1.377, Median: 0.995
  Up >=2x: 110, Down <=0.5x: 277
ACN3429: 2383 genes -> saved as 'berdl_fc_ACN3429_vs_ADP1'
  Mean FC: 1.318, Median: 1.005
  Up >=2x: 88, Down <=0.5x: 193
ACN3430: 2383 genes -> saved as 'berdl_fc_ACN3430_vs_ADP1'
  Mean FC: 1.371, Median: 0.953
  Up >=2x: 94, Down <=0.5x: 228

Saved 6 separate fold change files


## Load BERDL Fitness-Constrained Reference Flux

This step:
1. Loads the BERDL fitness-constrained flux result from ADP1BERDLFitnessFluxFitting
2. Extracts the flux distribution to use as the reference baseline
3. This flux was produced by fitting to TnSeq fitness + essentiality data on pyruvate media
4. The reference flux will be scaled by proteomics fold changes to compute target fluxes

In [3]:
%run util.py


# TODO (Phase 4c): Cross-notebook dependency — the reference flux comes from
# ADP1BERDLFitnessFluxFitting. That notebook has not been migrated yet.
# For now, attempt to load from cache; if not available, graceful None.
reference_flux = session.cache.load("berdl_fc_reference_flux", default=None)

if reference_flux is None:
    # Try loading from the legacy datacache directory as a fallback
    import json, os
    legacy_paths = [
        "datacache/ADP1BERDLFitnessFluxFitting/berdl_constrained_pyruvate_result.json",
        "datacache/berdl_constrained_pyruvate_result.json",
    ]
    for lp in legacy_paths:
        if os.path.exists(lp):
            with open(lp) as f:
                constrained_result = json.load(f)
            reference_flux = constrained_result.get("fluxes", {})
            session.cache.save("berdl_fc_reference_flux", reference_flux)
            print(f"Loaded reference flux from legacy path: {lp}")
            break

if reference_flux is not None:
    print(f"Loaded BERDL fitness-constrained reference flux:")
    print(f"  Total reactions with flux data: {len(reference_flux)}")
    print(f"  Non-zero fluxes: {sum(1 for v in reference_flux.values() if abs(v) > 1e-9)}")
else:
    print("WARNING: Reference flux not available.")
    print("Run ADP1BERDLFitnessFluxFitting notebook first, or wait for Phase 4c migration.")
    print("Downstream cells will detect None and skip cleanly.")

Loaded BERDL fitness-constrained reference flux:
  Total reactions with flux data: 1504
  Non-zero fluxes: 421


## Run Fold Change Flux Analysis for All Conditions

This step:
1. Caps reference flux at +/-20 to prevent extreme target fluxes
2. For each target condition (strain), computes fold changes relative to ADP1
3. Uses fit_flux_to_proteomics_fold_change_data to fit fluxes (fold changes capped at 3x up/down internally)
4. Uses the BERDL fitness-constrained flux as the reference distribution
5. Collects results including:
   - Fitted flux distributions
   - Reactions with significant flux changes
   - Proteins whose fold changes could/could not be implemented
   - Reaction matching statistics

In [4]:
%run util.py
import cobra.io
import copy
import cplex

from modelseedpy import MSExpression, MSModelUtil


# Ceilings for fold change and flux
MAX_FOLD_CHANGE = 3.0
MAX_FLUX = 20.0
LOG2_FC_CAP = np.log2(MAX_FOLD_CHANGE)

# Load saved data
strain_info = session.cache.load("berdl_fc_strains")
averaged_expression_data = session.cache.load("berdl_fc_averaged_expression")
reference_flux_raw = session.cache.load("berdl_fc_reference_flux", default=None)

if reference_flux_raw is None:
    print("ERROR: Reference flux not available. Skipping fold change analysis.")
    print("Run cell 3 first with valid reference flux data.")
else:
    # Cap reference flux at +/-MAX_FLUX
    reference_flux = {}
    capped_count = 0
    for rxn_id, flux in reference_flux_raw.items():
        if flux > MAX_FLUX:
            reference_flux[rxn_id] = MAX_FLUX
            capped_count += 1
        elif flux < -MAX_FLUX:
            reference_flux[rxn_id] = -MAX_FLUX
            capped_count += 1
        else:
            reference_flux[rxn_id] = flux

    print(f"Reference flux: {capped_count} reactions capped to +/-{MAX_FLUX}")

    # Load model
    model = cobra.io.load_json_model("models/MergedADP1Model.json")

    # === MEDIA FIX (2026-05-05) ===
    # TODO (Phase 4d / Phase 5): media should come from registered Sample.media
    # via Media.resolve_composition(session). For now, use the _legacy KBase
    # shim - all conditions in this notebook are pyruvate-based.
    from util_legacy import NotebookUtil
    _legacy = NotebookUtil()
    pyruvate_media = _legacy.get_media("KBaseMedia/Carbon-Pyruvic-Acid", msmedia=True)
    _legacy.set_media(model, pyruvate_media)
    # === END MEDIA FIX ===

    if "rxn01332_c0" in [r.id for r in model.reactions]:
        model.reactions.get_by_id("rxn01332_c0").lower_bound = 0
        model.reactions.get_by_id("rxn01332_c0").upper_bound = 0
    if "DgoA" in [r.id for r in model.reactions]:
        model.reactions.get_by_id("DgoA").lower_bound = -1000
        model.reactions.get_by_id("DgoA").upper_bound = 1000

    # Build expression DataFrame and cap log2 fold changes
    expression_df = pd.DataFrame.from_dict(averaged_expression_data)
    ref_col = strain_info["reference_condition"]
    target_conditions = strain_info["target_conditions"]

    genes_capped = 0
    for col in target_conditions:
        if col in expression_df.columns and ref_col in expression_df.columns:
            log2_fc = expression_df[col] - expression_df[ref_col]
            too_high = log2_fc > LOG2_FC_CAP
            too_low = log2_fc < -LOG2_FC_CAP
            genes_capped += int(too_high.sum() + too_low.sum())
            expression_df.loc[too_high, col] = expression_df.loc[too_high, ref_col] + LOG2_FC_CAP
            expression_df.loc[too_low, col] = expression_df.loc[too_low, ref_col] - LOG2_FC_CAP

    print(f"Expression data: {genes_capped} gene-condition pairs capped to +/-{LOG2_FC_CAP:.3f} log2 ({MAX_FOLD_CHANGE}x)")

    # Build MSExpression from the (log2) expression matrix.
    # MSExpression.from_dataframe expects feature IDs in the first column, not the index.
    expr_df_for_msexpr = expression_df.reset_index().rename(columns={"index": "gene_id"})
    expression = MSExpression.from_dataframe(
        expr_df_for_msexpr,
        genome_or_model=None,            # auto-create features from the gene_id column
        create_missing_features=True,
        type="Log2",
    )
    print(f"Built MSExpression: {len(expression.features)} features, {len(expression.conditions)} conditions")

    print(f"\nReference condition: {ref_col}")
    print(f"Analyzing {len(target_conditions)} target conditions")
    print("=" * 80)

    fold_change_results = {}

    for target_condition in target_conditions:
        short_name = target_condition.replace("Pyruvate_", "").replace("_DgoA_2025", "")
        print(f"\nProcessing: {short_name}")
        print("-" * 60)

        try:
            # Wrap a fresh copy of the model in MSModelUtil
            model_copy = copy.deepcopy(model)
            model_util = MSModelUtil.get(model_copy)

            # Canonical ModelSEEDpy fold-change-constrained flux fitting
            result = expression.fit_flux_to_proteomics_fold_change_data(
                model=model_util,
                reference_condition=ref_col,
                target_condition=target_condition,
                reference_flux=reference_flux,
            )

            sol = result["solution"]
            biomass = sol.fluxes.get("bio1", 0.0)
            n_active = int((sol.fluxes.abs() > 1e-9).sum())

            print(f"  Status: {sol.status}")
            print(f"  Biomass: {biomass:.6f}")
            print(f"  Active reactions: {n_active}")

            fold_change_results[target_condition] = {
                "fluxes": {k: v for k, v in sol.fluxes.items() if abs(v) > 1e-9},
                "biomass": biomass,
                "objective_value": sol.objective_value,
                "status": sol.status,
                "target_flux": result.get("target_flux", {}),
                "fold_changes": result.get("fold_changes", {}),
                "flux_changes": result.get("flux_changes", {}),
                "protein_changes": result.get("protein_changes", {}),
                "n_active_reactions": n_active,
            }
        except Exception as e:
            print(f"  ERROR: {e}")
            import traceback
            traceback.print_exc()
            fold_change_results[target_condition] = {
                "status": "error",
                "error": str(e),
            }

    # Save all results
    session.cache.save("ADP1BERDLFoldChangeAnalysis", fold_change_results)

    print("\n" + "=" * 80)
    print(f"Completed analysis for {len(fold_change_results)} conditions")
    print("Results saved via session.cache")


modelseedpy 0.4.2
Reference flux: 0 reactions capped to +/-20.0
/home/chenry/Dropbox/Projects/KBUtilLib/src


2026-05-06 21:33:18,235 - util_legacy.NotebookUtil - INFO - Loaded configuration from: /home/chenry/.kbutillib/config.yaml
2026-05-06 21:33:18,236 - util_legacy.NotebookUtil - INFO - Loaded kbase tokens from /home/chenry/.kbase/token


loading biochemistry database from /home/chenry/Dropbox/Projects/ModelSEEDDatabase


2026-05-06 21:33:26,027 - util_legacy.NotebookUtil - INFO - ModelSEED database loaded from /home/chenry/Dropbox/Projects/ModelSEEDDatabase
2026-05-06 21:33:26,700 - util_legacy.NotebookUtil - INFO - BLAST tools are available
2026-05-06 21:33:26,701 - util_legacy.NotebookUtil - INFO - Notebook name: ADP1BERDLFoldChangeAnalysis
2026-05-06 21:33:26,702 - util_legacy.NotebookUtil - INFO - Notebook environment detected
2026-05-06 21:33:26,703 - util_legacy.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/
2026-05-06 21:33:26,735 - util_legacy.NotebookUtil - INFO - AICurationUtils initialized with backend: argo
2026-05-06 21:33:26,739 - util_legacy.NotebookUtil - INFO - Loaded configuration from: /home/chenry/.kbutillib/config.yaml
2026-05-06 21:33:26,740 - util_legacy.NotebookUtil - INFO - Loaded kbase tokens from /home/chenry/.kbase/token
2026-05-06 21:33:26,742 - util_legacy.

cobrakbase 0.4.0


2026-05-06 21:33:27,272 - util_legacy.NotebookUtil - INFO - BLAST tools are available
2026-05-06 21:33:27,273 - util_legacy.NotebookUtil - INFO - Notebook name: ADP1BERDLFoldChangeAnalysis
2026-05-06 21:33:27,274 - util_legacy.NotebookUtil - INFO - Notebook environment detected
2026-05-06 21:33:27,274 - util_legacy.NotebookUtil - INFO - ArgoGatewayClient initialised | model=gpto3mini env=dev timeout=120.0s url=https://apps-dev.inside.anl.gov/argoapi/api/v1/resource/streamchat/
2026-05-06 21:33:27,279 - util_legacy.NotebookUtil - INFO - AICurationUtils initialized with backend: argo


Expression data: 733 gene-condition pairs capped to +/-1.585 log2 (3.0x)
Built MSExpression: 2383 features, 7 conditions

Reference condition: Pyruvate_ADP1_DgoA_2025
Analyzing 6 target conditions

Processing: ACN2586
------------------------------------------------------------
  Status: optimal
  Biomass: 0.117852
  Active reactions: 430

Processing: ACN2821
------------------------------------------------------------
  Status: optimal
  Biomass: 0.105606
  Active reactions: 456

Processing: ACN3425
------------------------------------------------------------
  Status: optimal
  Biomass: 0.119537
  Active reactions: 439

Processing: ACN3427
------------------------------------------------------------
  Status: optimal
  Biomass: 0.125670
  Active reactions: 429

Processing: ACN3429
------------------------------------------------------------
  Status: optimal
  Biomass: 0.123098
  Active reactions: 440

Processing: ACN3430
------------------------------------------------------------
 

## Summary Statistics

This step:
1. Loads the fold change analysis results
2. Generates summary tables comparing all conditions
3. Identifies common patterns across strains

In [5]:
%run util.py


results = session.cache.load("ADP1BERDLFoldChangeAnalysis", default=None)
if results is None:
    print("No results available yet. Run cell 4 first.")
else:
    print("BERDL FOLD CHANGE ANALYSIS SUMMARY")
    print("Reference flux: BERDL fitness-constrained (pyruvate)")
    print("=" * 100)

    summary_data = []
    for condition, data in results.items():
        short = condition.replace("Pyruvate_", "").replace("_DgoA_2025", "")
        if data.get("status") == "error":
            summary_data.append({"Condition": short, "Status": "ERROR"})
        else:
            summary_data.append({
                "Condition": short,
                "Status": data.get("status", "unknown"),
                "Biomass": f"{data.get('biomass', 0):.6f}",
                "Active Rxns": data.get("n_active_reactions", 0),
            })

    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))

    # Save summary to Excel
    output_dir = "nboutput/ADP1BERDLFoldChangeAnalysis"
    os.makedirs(output_dir, exist_ok=True)
    summary_df.to_excel(f"{output_dir}/berdl_fold_change_summary.xlsx", index=False)
    print(f"\nSummary saved to {output_dir}/berdl_fold_change_summary.xlsx")

BERDL FOLD CHANGE ANALYSIS SUMMARY
Reference flux: BERDL fitness-constrained (pyruvate)
Condition  Status  Biomass  Active Rxns
  ACN2586 optimal 0.117852          430
  ACN2821 optimal 0.105606          456
  ACN3425 optimal 0.119537          439
  ACN3427 optimal 0.125670          429
  ACN3429 optimal 0.123098          440
  ACN3430 optimal 0.090903          462

Summary saved to nboutput/ADP1BERDLFoldChangeAnalysis/berdl_fold_change_summary.xlsx


## Protein Implementation Analysis

This step:
1. Identifies proteins with significant fold changes that could not be implemented
2. Analyzes why certain fold changes couldn't be matched in the flux solution
3. Identifies common bottlenecks across strains

In [6]:
%run util.py


results = session.cache.load("ADP1BERDLFoldChangeAnalysis", default=None)
if results is None:
    print("No results available yet. Run cell 4 first.")
else:
    print("FOLD CHANGE ANALYSIS — PER-CONDITION SUMMARY")
    print("=" * 80)

    for condition, data in results.items():
        short = condition.replace("Pyruvate_", "").replace("_DgoA_2025", "")
        if data.get("status") == "error":
            print(f"\n{short}: ERROR — {data.get('error', 'unknown')}")
            continue

        n_genes = len(data.get("gene_fold_changes", {}))
        n_target = len(data.get("target_flux", {}))
        print(f"\n{short}:")
        print(f"  Genes with fold change: {n_genes}")
        print(f"  Reactions with target flux: {n_target}")
        print(f"  Biomass: {data.get('biomass', 0):.6f}")
        print(f"  Active reactions: {data.get('n_active_reactions', 0)}")

FOLD CHANGE ANALYSIS — PER-CONDITION SUMMARY

ACN2586:
  Genes with fold change: 0
  Reactions with target flux: 2
  Biomass: 0.117852
  Active reactions: 430

ACN2821:
  Genes with fold change: 0
  Reactions with target flux: 2
  Biomass: 0.105606
  Active reactions: 456

ACN3425:
  Genes with fold change: 0
  Reactions with target flux: 2
  Biomass: 0.119537
  Active reactions: 439

ACN3427:
  Genes with fold change: 0
  Reactions with target flux: 2
  Biomass: 0.125670
  Active reactions: 429

ACN3429:
  Genes with fold change: 0
  Reactions with target flux: 2
  Biomass: 0.123098
  Active reactions: 440

ACN3430:
  Genes with fold change: 0
  Reactions with target flux: 2
  Biomass: 0.090903
  Active reactions: 462


## Compare with Lactate-Based Fold Change Analysis

**DEPRECATED (Phase 4b):** The non-BERDL lactate notebook is being deprecated in Phase 4d.
The BERDL fitness-constrained flux supersedes the lactate reference condition.
This comparison is no longer applicable.

In [7]:
# [deferred — non-BERDL lactate notebook is being deprecated in Phase 4d]
#
# The comparison with ADP1FoldChangeAnalysis (lactate-based) is no longer
# applicable post-migration. The BERDL fitness-constrained flux supersedes
# the lactate reference. This cell is intentionally left as a placeholder.

%run util.py
print("[deferred — non-BERDL lactate notebook is being deprecated in Phase 4d]")

[deferred — non-BERDL lactate notebook is being deprecated in Phase 4d]


## Visualize Reference Flux on Escher Map

This step:
1. Loads the BERDL fitness-constrained reference flux
2. Visualizes it on the full.json Escher map
3. Generates an interactive HTML file for exploring the flux distribution

In [10]:
%run util.py


reference_flux = session.cache.load("berdl_fc_reference_flux", default=None)

if reference_flux is None:
    print("Reference flux not available. Skipping visualization.")
else:
    import cobra.io
    model = cobra.io.load_json_model("models/MergedADP1Model.json")

    print(f"BERDL reference flux has {len(reference_flux)} reactions")
    print(f"Non-zero fluxes: {sum(1 for v in reference_flux.values() if abs(v) > 1e-9)}")

    output_dir = "nboutput/ADP1BERDLFoldChangeAnalysis"
    os.makedirs(output_dir, exist_ok=True)

    output_file = _legacy.create_map_html2(
            model=model,
            flux=pd.Series(reference_flux),
            map="full",
            output_path=f"{output_dir}/escher_berdl_reference_flux.html",
        )

    print(f"\nReference flux map saved to: {output_file}")

2026-05-06 21:50:30,622 - util_legacy.NotebookUtil - INFO - Loaded map 'full' from local index: /home/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-05-06 21:50:30,628 - util_legacy.NotebookUtil - INFO - Updated names for 759 reactions in map


BERDL reference flux has 1504 reactions
Non-zero fluxes: 421

Reference flux map saved to: nboutput/ADP1BERDLFoldChangeAnalysis/escher_berdl_reference_flux.html


## Visualize Fitted Flux for Selected Condition

This step:
1. Allows selection of a target condition to visualize
2. Loads the fitted flux from fold change analysis
3. Displays on Escher map to compare with reference

In [11]:
%run util.py


results = session.cache.load("ADP1BERDLFoldChangeAnalysis", default=None)

# Select condition to visualize
selected_condition = "Pyruvate_ACN2586_DgoA_2025"
short_name = selected_condition.replace("Pyruvate_", "").replace("_DgoA_2025", "")

if results is None:
    print("No results available yet. Run cell 4 first.")
else:
    print(f"Selected condition: {short_name}")
    condition_data = results[selected_condition]

    if condition_data.get("status") == "error":
        print(f"ERROR: {condition_data.get('error', 'Unknown')}")
    else:
        fitted_flux = condition_data["fluxes"]
        print(f"Fitted flux has {len(fitted_flux)} reactions")
        print(f"Non-zero fluxes: {sum(1 for v in fitted_flux.values() if abs(v) > 1e-9)}")

        import cobra.io
        model = cobra.io.load_json_model("models/MergedADP1Model.json")

        output_dir = "nboutput/ADP1BERDLFoldChangeAnalysis"
        os.makedirs(output_dir, exist_ok=True)
        output_file = _legacy.create_map_html2(
            model=model,
            flux=pd.Series(fitted_flux),
            map="full",
            output_path=f"{output_dir}/escher_berdl_fitted_flux_{short_name}.html",
        )
        print(f"\nFitted flux map saved to: {output_file}")

Selected condition: ACN2586
Fitted flux has 430 reactions
Non-zero fluxes: 430


2026-05-06 21:52:36,547 - util_legacy.NotebookUtil - INFO - Loaded map 'full' from local index: /home/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-05-06 21:52:36,551 - util_legacy.NotebookUtil - INFO - Updated names for 759 reactions in map



Fitted flux map saved to: nboutput/ADP1BERDLFoldChangeAnalysis/escher_berdl_fitted_flux_ACN2586.html


## Visualize Flux Differential (Fitted - Reference)

This step:
1. Computes the flux differential between fitted and BERDL reference fluxes
2. Visualizes on Escher map using directional color scheme
3. Positive values (green) = increased flux, Negative values (red) = decreased flux

In [12]:
%run util.py


results = session.cache.load("ADP1BERDLFoldChangeAnalysis", default=None)
reference_flux = session.cache.load("berdl_fc_reference_flux", default=None)

selected_condition = "Pyruvate_ACN2586_DgoA_2025"
short_name = selected_condition.replace("Pyruvate_", "").replace("_DgoA_2025", "")

if results is None or reference_flux is None:
    print("Results or reference flux not available. Skipping.")
else:
    print(f"Computing flux differential for: {short_name}")
    condition_data = results[selected_condition]

    if condition_data.get("status") == "error":
        print(f"ERROR: {condition_data.get('error', 'Unknown')}")
    else:
        fitted_flux = condition_data["fluxes"]
        flux_differential = {}
        all_rxns = set(fitted_flux.keys()) | set(reference_flux.keys())
        for rxn_id in all_rxns:
            diff = fitted_flux.get(rxn_id, 0) - reference_flux.get(rxn_id, 0)
            flux_differential[rxn_id] = diff

        positive_changes = sum(1 for v in flux_differential.values() if v > 0.01)
        negative_changes = sum(1 for v in flux_differential.values() if v < -0.01)
        unchanged = len(flux_differential) - positive_changes - negative_changes

        print(f"\nFlux differential statistics:")
        print(f"  Increased (>0.01): {positive_changes} reactions")
        print(f"  Decreased (<-0.01): {negative_changes} reactions")
        print(f"  Unchanged: {unchanged} reactions")

        sorted_diffs = sorted(flux_differential.items(), key=lambda x: abs(x[1]), reverse=True)
        print(f"\nTop 10 largest flux changes:")
        for rxn_id, diff in sorted_diffs[:10]:
            direction = "+" if diff > 0 else ""
            print(f"  {rxn_id}: {direction}{diff:.4f}")

        import cobra.io
        model = cobra.io.load_json_model("models/MergedADP1Model.json")

        output_dir = "nboutput/ADP1BERDLFoldChangeAnalysis"
        os.makedirs(output_dir, exist_ok=True)
        output_file = _legacy.create_map_html2(
            model=model,
            flux=pd.Series(flux_differential),
            map="full",
            output_path=f"{output_dir}/escher_berdl_flux_diff_{short_name}.html",
        )
        print(f"\nFlux differential map saved to: {output_file}")

Computing flux differential for: ACN2586

Flux differential statistics:
  Increased (>0.01): 59 reactions
  Decreased (<-0.01): 48 reactions
  Unchanged: 1397 reactions

Top 10 largest flux changes:
  rxn14418_c0: +5.9668
  rxn14419_c0: +5.9486
  rxn08173_c0: +4.6101
  rxn00154_c0: -4.1012
  rxn01241_c0: +3.4713
  rxn05488_c0: +3.4562
  EX_cpd00029_e0: -3.4562
  rxn00172_c0: +3.4015
  EX_cpd00001_e0: +2.9206
  rxn05319_c0: -2.9206


2026-05-06 21:54:25,864 - util_legacy.NotebookUtil - INFO - Loaded map 'full' from local index: /home/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-05-06 21:54:25,869 - util_legacy.NotebookUtil - INFO - Updated names for 759 reactions in map



Flux differential map saved to: nboutput/ADP1BERDLFoldChangeAnalysis/escher_berdl_flux_diff_ACN2586.html


## Flux Differential Dictionary for Manual Escher Rendering

This step:
1. Computes the flux differential (fitted - reference) for a selected condition
2. Filters to only non-trivial changes (|diff| > 0.01)
3. Prints the dictionary in a format suitable for copy-pasting into Escher's reaction data input

In [13]:
%run util.py


results = session.cache.load("ADP1BERDLFoldChangeAnalysis", default=None)
reference_flux = session.cache.load("berdl_fc_reference_flux", default=None)

selected_condition = "Pyruvate_ACN2586_DgoA_2025"
short_name = selected_condition.replace("Pyruvate_", "").replace("_DgoA_2025", "")

if results is None or reference_flux is None:
    print("Results or reference flux not available. Skipping.")
else:
    print(f"Flux differential dictionary for: {short_name}")
    print(f"Format: reaction_id -> (fitted - reference) flux value")
    print(f"Only showing |diff| > 0.01")
    print("=" * 60)

    condition_data = results[selected_condition]
    fitted_flux = condition_data["fluxes"]

    flux_diff_dict = {}
    all_rxns = set(fitted_flux.keys()) | set(reference_flux.keys())
    for rxn_id in sorted(all_rxns):
        fitted_val = abs(fitted_flux.get(rxn_id, 0))
        ref_val = abs(reference_flux.get(rxn_id, 0))
        if ref_val > 0.000001 and fitted_val > 0.000001:
            diff = math.log2(fitted_val / ref_val)
            diff = max(-1.0, min(1.0, diff))
            diff = diff + 1
            flux_diff_dict[rxn_id] = diff
            flux_diff_dict[rxn_id + "-rev"] = diff

    print(f"\n{len(flux_diff_dict)} reactions with non-trivial flux changes:")
    print(f"\nTotal: {len(flux_diff_dict)} reactions")
    print(f"Increased: {sum(1 for v in flux_diff_dict.values() if v > 0)}")
    print(f"Decreased: {sum(1 for v in flux_diff_dict.values() if v < 0)}")
    session.cache.save("flux_diff", flux_diff_dict)

Flux differential dictionary for: ACN2586
Format: reaction_id -> (fitted - reference) flux value
Only showing |diff| > 0.01

758 reactions with non-trivial flux changes:

Total: 758 reactions
Increased: 748
Decreased: 0


## Generate All Condition Maps

This step:
1. Generates Escher maps for all conditions in a single batch
2. Creates fitted flux maps and flux differential maps for each strain
3. Provides links to all generated visualizations

In [14]:
%run util.py


results = session.cache.load("ADP1BERDLFoldChangeAnalysis", default=None)
reference_flux = session.cache.load("berdl_fc_reference_flux", default=None)

if results is None or reference_flux is None:
    print("Results or reference flux not available. Skipping batch map generation.")
else:
    import cobra.io
    model = cobra.io.load_json_model("models/MergedADP1Model.json")

    print("Generating Escher maps for all conditions...")
    print("=" * 80)

    output_dir = "nboutput/ADP1BERDLFoldChangeAnalysis"
    os.makedirs(output_dir, exist_ok=True)

    generated_files = {"reference": None, "fitted": {}, "differential": {}}

    # Reference flux map
    print("\n[Reference Flux - BERDL]")
    ref_output = _legacy.create_map_html2(
            model=model,
            flux=pd.Series(reference_flux),
            map="full",
            output_path=f"{output_dir}/escher_berdl_reference_flux.html",
        )
    generated_files["reference"] = str(ref_output)
    print(f"  Saved: {ref_output}")

    # Maps for each condition
    for condition, data in results.items():
        short_name = condition.replace("Pyruvate_", "").replace("_DgoA_2025", "")
        print(f"\n[{short_name}]")

        if data.get("status") == "error":
            print("  SKIPPED — Error in analysis")
            continue

        fitted_flux = data["fluxes"]

        # Fitted flux map
        fitted_output = _legacy.create_map_html2(
            model=model,
            flux=pd.Series(fitted_flux),
            map="full",
            output_path=f"{output_dir}/escher_berdl_fitted_flux_{short_name}.html",
        )
        generated_files["fitted"][short_name] = str(fitted_output)
        print(f"  Fitted flux: {fitted_output}")

        # Differential map
        flux_differential = {}
        all_rxns = set(fitted_flux.keys()) | set(reference_flux.keys())
        for rxn_id in all_rxns:
            diff = fitted_flux.get(rxn_id, 0) - reference_flux.get(rxn_id, 0)
            flux_differential[rxn_id] = diff

        diff_output = _legacy.create_map_html2(
            model=model,
            flux=pd.Series(flux_differential),
            map="full",
            output_path=f"{output_dir}/escher_berdl_flux_diff_{short_name}.html",
        )
        generated_files["differential"][short_name] = str(diff_output)
        print(f"  Differential: {diff_output}")

    session.cache.save("berdl_fold_change_escher_files", generated_files)

    print("\n" + "=" * 80)
    print("All maps generated successfully!")
    n_fitted = len(generated_files["fitted"])
    n_diff = len(generated_files["differential"])
    print(f"Total files: 1 reference + {n_fitted} fitted + {n_diff} differential")

2026-05-06 21:55:53,245 - util_legacy.NotebookUtil - INFO - Loaded map 'full' from local index: /home/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-05-06 21:55:53,254 - util_legacy.NotebookUtil - INFO - Updated names for 759 reactions in map


Generating Escher maps for all conditions...

[Reference Flux - BERDL]


2026-05-06 21:57:08,395 - util_legacy.NotebookUtil - INFO - Loaded map 'full' from local index: /home/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-05-06 21:57:08,400 - util_legacy.NotebookUtil - INFO - Updated names for 759 reactions in map


  Saved: nboutput/ADP1BERDLFoldChangeAnalysis/escher_berdl_reference_flux.html

[ACN2586]


2026-05-06 21:58:33,766 - util_legacy.NotebookUtil - INFO - Loaded map 'full' from local index: /home/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-05-06 21:58:33,770 - util_legacy.NotebookUtil - INFO - Updated names for 759 reactions in map


...

  Fitted flux: nboutput/ADP1BERDLFoldChangeAnalysis/escher_berdl_fitted_flux_ACN3429.html


2026-05-06 22:07:15,156 - util_legacy.NotebookUtil - INFO - Loaded map 'full' from local index: /home/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-05-06 22:07:15,163 - util_legacy.NotebookUtil - INFO - Updated names for 759 reactions in map


  Differential: nboutput/ADP1BERDLFoldChangeAnalysis/escher_berdl_flux_diff_ACN3429.html

[ACN3430]


2026-05-06 22:07:38,110 - util_legacy.NotebookUtil - INFO - Loaded map 'full' from local index: /home/chenry/Dropbox/Projects/KBUtilLib/src/kbutillib/../../data/escher_maps/full.json
2026-05-06 22:07:38,114 - util_legacy.NotebookUtil - INFO - Updated names for 759 reactions in map


  Fitted flux: nboutput/ADP1BERDLFoldChangeAnalysis/escher_berdl_fitted_flux_ACN3430.html
  Differential: nboutput/ADP1BERDLFoldChangeAnalysis/escher_berdl_flux_diff_ACN3430.html

All maps generated successfully!
Total files: 1 reference + 6 fitted + 6 differential
